In [1]:
import pandas as pd

In [2]:
neurons = pd.read_csv("../data/neurons.csv.gz")
print(neurons.shape)
neurons.head()

(139255, 10)


,root_id,group,nt_type,nt_type_score,da_avg,ser_avg,gaba_avg,glut_avg,ach_avg,oct_avg
0,720575940596125868,LO.LOP,ACH,0.57,0.03,0.00,0.05,0.28,0.57,0.07
1,720575940597856265,ME,ACH,0.85,0.01,0.00,0.03,0.04,0.85,0.07
2,720575940597944841,ME.LO,ACH,0.82,0.02,0.00,0.01,0.05,0.82,0.09
3,720575940598267657,ME,GABA,0.74,0.01,0.01,0.74,0.19,0.03,0.02
4,720575940599333574,ME,ACH,0.61,0.02,0.00,0.13,0.13,0.61,0.11


In [3]:
classification = pd.read_csv("../data/classification.csv.gz")
print(classification.shape)
cell_types=pd.read_csv("../data/consolidated_cell_types.csv.gz")
print(cell_types.shape)
cell_stats=pd.read_csv("../data/cell_stats.csv.gz")
print(cell_stats.shape)
connections=pd.read_csv("../data/connections_princeton.csv.gz")
print(connections.shape)

(139255, 8)
(138327, 3)
(139246, 4)
(5342446, 5)


In [4]:
classification.head()

,root_id,flow,super_class,class,sub_class,hemilineage,side,nerve
0,720575940599457990,intrinsic,optic,optic_lobe_intrinsic,t4_neuron,NaN,right,NaN
1,720575940599763910,intrinsic,optic,optic_lobe_intrinsic,t5_neuron,NaN,right,NaN
2,720575940600623020,intrinsic,optic,optic_lobe_intrinsic,distal_medulla,NaN,right,NaN
3,720575940602309600,afferent,sensory_ascending,AN,SA_DMT_DMetaN,NaN,right,CV
4,720575940602564320,intrinsic,central,Kenyon_Cell,KCg,MBp4,right,NaN


In [5]:
cell_types.head()

,root_id,primary_type,additional_type(s)
0,720575940599457990,T4b,NaN
1,720575940599763910,T5b,NaN
2,720575940600623020,Dm3q,NaN
3,720575940602564320,KCg-m,NaN
4,720575940602720940,JO-CA,NaN


In [6]:
cell_stats.head()

,root_id,length_nm,area_nm,size_nm
0,720575940596125868,396840,628299136,37616302080
1,720575940597856265,1184013,1634424448,100004290560
2,720575940597944841,784742,1022307072,58954229760
3,720575940598267657,1645960,2346787840,140744601600
4,720575940599333574,575383,1366784768,111170764800


In [7]:
connections.head()

,pre_root_id,post_root_id,neuropil,syn_count,nt_type
0,720575940625363947,720575940623224444,ME_L,12,GABA
1,720575940630432382,720575940618518557,ME_L,67,ACH
2,720575940627314521,720575940626337738,ME_L,10,GABA
3,720575940620280405,720575940620204726,ME_L,15,ACH
4,720575940636942447,720575940613789411,LA_L,1,GLUT


In [8]:
y=classification['super_class'].values

In [9]:
ids = classification["root_id"]
in_partners = connections.groupby("post_root_id")["pre_root_id"].nunique()
out_partners = connections.groupby("pre_root_id")["post_root_id"].nunique()
in_syn = connections.groupby("post_root_id")["syn_count"].sum()
out_syn = connections.groupby("pre_root_id")["syn_count"].sum()
# 1. Συνάψεις ανά (νευρώνα, νευροδιαβιβαστή)
nt_counts = connections.groupby(["pre_root_id", "nt_type"])["syn_count"].sum()

# 2. Μία γραμμή ανά νευρώνα, μία στήλη ανά νευροδιαβιβαστή (κενά = 0)
nt_frac = nt_counts.unstack(fill_value=0)

# 3. Από πλήθη σε ποσοστά: κάθε γραμμή διαιρείται με το σύνολό της
nt_frac = nt_frac.div(nt_frac.sum(axis=1), axis=0)

# 4. Ίδια σειρά με το classification, ώστε να ταιριάζει με το y
nt_frac = nt_frac.reindex(classification["root_id"], fill_value=0).add_prefix("out_nt_")

nt_frac.head()




nt_type,out_nt_ACH,out_nt_DA,out_nt_GABA,out_nt_GLUT,out_nt_OCT,out_nt_SER
root_id,,,,,,
720575940599457990,1.0,0.0,0.0,0.0,0.0,0.0
720575940599763910,1.0,0.0,0.0,0.0,0.0,0.0
720575940600623020,0.0,0.0,0.0,1.0,0.0,0.0
720575940602309600,1.0,0.0,0.0,0.0,0.0,0.0
720575940602564320,1.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# Χαρακτηριστικά συνδεσιμότητας (0 αν ο νευρώνας δεν έχει συνδέσεις)
conn = pd.DataFrame({
    "in_partners": in_partners,
    "out_partners": out_partners,
    "in_syn": in_syn,
    "out_syn": out_syn,
}).reindex(ids).fillna(0)

# Μέγεθος από το cell_stats (μόνο αριθμητικές στήλες)
stats = cell_stats.set_index("root_id").select_dtypes("number").reindex(ids)

# Όλα μαζί
X = pd.concat([conn, stats, nt_frac], axis=1)
X.head()

,in_partners,out_partners,in_syn,out_syn,length_nm,area_nm,size_nm,out_nt_ACH,out_nt_DA,out_nt_GABA,out_nt_GLUT,out_nt_OCT,out_nt_SER
root_id,,,,,,,,,,,,,
720575940599457990,7.0,13.0,75.0,139.0,358695.0,6.707910e+08,4.897548e+10,1.0,0.0,0.0,0.0,0.0,0.0
720575940599763910,13.0,16.0,151.0,162.0,395365.0,6.452087e+08,4.382769e+10,1.0,0.0,0.0,0.0,0.0,0.0
720575940600623020,13.0,5.0,112.0,29.0,361217.0,5.845093e+08,4.724302e+10,0.0,0.0,0.0,1.0,0.0,0.0
720575940602309600,3.0,5.0,17.0,27.0,108781.0,3.057528e+08,1.720609e+10,1.0,0.0,0.0,0.0,0.0,0.0
720575940602564320,8.0,8.0,141.0,72.0,657447.0,1.316774e+09,1.021091e+11,1.0,0.0,0.0,0.0,0.0,0.0


In [11]:
y.shape

(139255,)

In [12]:
X.shape

(139255, 13)

In [13]:
mask=X.notna().all(axis=1)
X=X[mask]
y=y[mask]

In [14]:
y.shape

(139246,)

In [15]:
X.shape

(139246, 13)

In [16]:
pd.Series(y).value_counts(dropna=False)

optic                 77868
central               32381
sensory               16935
visual_projection      7684
ascending              1750
descending             1305
sensory_ascending       612
visual_centrifugal      522
motor                   109
endocrine                80
Name: count, dtype: int64

In [17]:
pd.Series(y).value_counts(normalize=True)

optic                 0.559212
central               0.232545
sensory               0.121619
visual_projection     0.055183
ascending             0.012568
descending            0.009372
sensory_ascending     0.004395
visual_centrifugal    0.003749
motor                 0.000783
endocrine             0.000575
Name: proportion, dtype: float64

In [18]:
import numpy as np
from sklearn.model_selection import train_test_split


In [19]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

In [20]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [21]:
pd.Series(y_train).value_counts(normalize=True)

optic                 0.559217
central               0.232549
sensory               0.121614
visual_projection     0.055185
ascending             0.012568
descending            0.009377
sensory_ascending     0.004391
visual_centrifugal    0.003745
motor                 0.000780
endocrine             0.000575
Name: proportion, dtype: float64

In [22]:
pd.Series(y_val).value_counts(normalize=True)

optic                 0.559200
central               0.232537
sensory               0.121607
visual_projection     0.055202
ascending             0.012592
descending            0.009384
sensory_ascending     0.004405
visual_centrifugal    0.003734
motor                 0.000766
endocrine             0.000575
Name: proportion, dtype: float64

In [23]:
pd.Series(y_test).value_counts(normalize=True)

optic                 0.559200
central               0.232537
sensory               0.121655
visual_projection     0.055154
ascending             0.012544
descending            0.009336
sensory_ascending     0.004405
visual_centrifugal    0.003782
motor                 0.000814
endocrine             0.000575
Name: proportion, dtype: float64

In [24]:
y_train = pd.Series(y_train)
y_val = pd.Series(y_val)
y_test = pd.Series(y_test)

In [25]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [26]:
X_train.to_csv("../data/processed/X_train.csv")
X_val.to_csv("../data/processed/X_val.csv")
X_test.to_csv("../data/processed/X_test.csv")

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_val.to_csv("../data/processed/y_val.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [27]:
connections["neuropil"].nunique()

79

In [28]:
connections["neuropil"].value_counts()

neuropil
ME_R     660198
ME_L     616856
LO_R     316692
LO_L     306759
GNG      226326
          ...  
AME_R      1882
GA_R       1665
OCG        1582
BU_R       1469
BU_L        984
Name: count, Length: 79, dtype: int64

In [ ]:
out_np = connections.pivot_table(
    index="pre_root_id",
    columns="neuropil",
    values="syn_count",
    aggfunc="sum",
    fill_value=0
)
out_np.head()

neuropil,AL_L,AL_R,AME_L,AME_R,AMMC_L,AMMC_R,AOTU_L,AOTU_R,ATL_L,ATL_R,...,SLP_R,SMP_L,SMP_R,SPS_L,SPS_R,UNASGD,VES_L,VES_R,WED_L,WED_R
pre_root_id,,,,,,,,,,,,,,,,,,,,,
720575940596125868,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940597856265,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940597944841,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940598267657,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940599333574,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [31]:
out_np.shape

(137518, 79)

In [34]:
row=out_np.iloc[0]
row[row > 0]

neuropil
LOP_R    169
LO_R       5
Name: 720575940596125868, dtype: int64

In [36]:
out_frac = out_np.div(out_np.sum(axis=1), axis=0)
out_frac.iloc[0][out_frac.iloc[0] > 0]

neuropil
LOP_R    0.971264
LO_R     0.028736
Name: 720575940596125868, dtype: float64

In [38]:
out_frac.sum(axis=1).head()

pre_root_id
720575940596125868    1.0
720575940597856265    1.0
720575940597944841    1.0
720575940598267657    1.0
720575940599333574    1.0
dtype: float64

In [39]:
in_np = connections.pivot_table(
    index="post_root_id",
    columns="neuropil",
    values="syn_count",
    aggfunc="sum",
    fill_value=0
)
in_np.head()

neuropil,AL_L,AL_R,AME_L,AME_R,AMMC_L,AMMC_R,AOTU_L,AOTU_R,ATL_L,ATL_R,...,SLP_R,SMP_L,SMP_R,SPS_L,SPS_R,UNASGD,VES_L,VES_R,WED_L,WED_R
post_root_id,,,,,,,,,,,,,,,,,,,,,
720575940596125868,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940597856265,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940597944841,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940598267657,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
720575940599333574,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [41]:
in_frac = in_np.div(in_np.sum(axis=1), axis=0)
in_frac.iloc[0][in_frac.iloc[0] > 0]

neuropil
LOP_R    0.100775
LO_R     0.899225
Name: 720575940596125868, dtype: float64

In [42]:
in_frac.sum(axis=1).head()

post_root_id
720575940596125868    1.0
720575940597856265    1.0
720575940597944841    1.0
720575940598267657    1.0
720575940599333574    1.0
dtype: float64

In [43]:
out_frac = out_frac.add_prefix("out_")
in_frac = in_frac.add_prefix("in_")

In [45]:
out_frac.columns[:5]

Index(['out_AL_L', 'out_AL_R', 'out_AME_L', 'out_AME_R', 'out_AMMC_L'], dtype='str', name='neuropil')

In [46]:
in_frac.columns[:5]

Index(['in_AL_L', 'in_AL_R', 'in_AME_L', 'in_AME_R', 'in_AMMC_L'], dtype='str', name='neuropil')

In [47]:
out_frac.shape

(137518, 79)

In [48]:
in_frac.shape

(130183, 79)

In [49]:
np_feats = out_frac.join(in_frac, how="outer").fillna(0)

In [50]:
np_feats.shape

(138584, 158)

In [53]:
for split in ["train", "val", "test"]:
    X = pd.read_csv(f"../data/processed/X_{split}.csv")
    X_new = X.merge(np_feats, left_on="root_id", right_index=True, how="left").fillna(0)
    print(split, X.shape, "→", X_new.shape)
    X_new.to_csv(f"../data/processed/X_{split}_np.csv", index=False)

train (97472, 14) → (97472, 172)
val (20887, 14) → (20887, 172)
test (20887, 14) → (20887, 172)
